In [1]:
import os
import pandas as pd

os.makedirs('work/outputs', exist_ok=True)
df = pd.read_csv('/content/content_refresh_anonymized.csv')
print(df.shape)

(30000, 44)


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Softwareguy786/ML_INTERNSHIP_01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1a. Signal check #1 — Staleness

**Hypothesis:** pages that haven't been updated in a long time are more likely to be declining.

I bucket `days_since_last_update` into five groups, print `n` per bucket, and look at mean `trend_pct` and the share of pages with negative trend.

In [2]:
bins = [0, 30, 90, 180, 365, float('inf')]
labels = ['0-30', '31-90', '91-180', '181-365', '365+']

staleness_buckets = (
    df.assign(bucket=pd.cut(df['days_since_last_update'], bins=bins, labels=labels))
      .groupby('bucket', observed=True)
      .agg(
          n=('content_id', 'count'),
          mean_trend_pct=('trend_pct', 'mean'),
          pct_negative_trend=('trend_pct', lambda s: (s < 0).mean() * 100),
      )
      .round(2)
)

staleness_buckets

,n,mean_trend_pct,pct_negative_trend
bucket,,,
0-30,20480,0.78,61.42
31-90,175,-7.37,65.71
91-180,9171,-15.68,75.59
181-365,169,-4.72,51.48
365+,5,-96.17,60.00


### 1b. Signal check #2 — Volume

**Hypothesis:** high-impression pages carry more risk when they decline, so volume matters for prioritization.

I bucket by the existing `impression_tier` column, print `n` per bucket, and look at mean `trend_pct` and the share of pages with negative trend.

In [3]:
volume_buckets = (
    df.groupby('impression_tier', observed=True)
      .agg(
          n=('content_id', 'count'),
          mean_trend_pct=('trend_pct', 'mean'),
          pct_negative_trend=('trend_pct', lambda s: (s < 0).mean() * 100),
      )
      .round(2)
      .sort_values('n', ascending=False)
)

volume_buckets

,n,mean_trend_pct,pct_negative_trend
impression_tier,,,
low,11248,0.27,49.68
moderate,10469,-2.77,74.55
good,7205,-16.99,76.82
excellent,1078,19.63,73.01


## 1. My rule and its reason codes

### Signal verdicts

**Staleness — MIXED.** Not monotonic. Pages 91–180 days stale have the worst mean `trend_pct` (-15.68) and highest share declining (75.6%), supporting the hypothesis. But 181–365 days stale look *better* than fresh (51.5% vs 61.4% negative), and 365+ has only n=5 (noise). Conclusion: staleness matters up to ~180 days, then flattens. The rule caps staleness at 180 days.

**Volume — MIXED.** % declining rises from low (49.7%) → moderate (74.6%) → good (76.8%), supporting the hypothesis. But `excellent` pages have a *positive* mean trend (+19.63) — the top tier is growing. Conclusion: volume signals *where* risk concentrates, not raw decline. The rule multiplies volume by decline so volume only contributes when the page is actually declining.

Both signals are flag-linked (staleness → refresh flag, volume → quick-win flag). Both weights are informed by what the tables showed, not by what the hypothesis wanted.

### Rule in plain words

Each page gets a **refresh urgency score** in [0, 100] = weighted sum of three signals:

1. **Decline (weight 0.40):** how negative `trend_pct` is, capped at 100. Positive-trend pages contribute 0.
2. **Staleness (weight 0.30):** `days_since_last_update` scaled 0–100, **capped at 180 days** — because the check showed the effect flattens past ~180.
3. **Volume-at-risk (weight 0.30):** log-scaled `impressions_90d`, **only counted if `trend_pct < 0`** — because a decline on a high-volume page hurts most, but high volume alone doesn't predict decline.

Each page gets exactly **one reason code** (dominant driver):
- `DECLINE` — decline contributes ≥50% of score
- `STALE` — staleness contributes ≥50%
- `VOLUME_AT_RISK` — volume-at-risk contributes ≥50%
- `MIXED` — no single signal reaches 50%
- `NO_SIGNAL` — all signals are 0

Each page gets one **action label** by score rank:
- `REFRESH_NOW` — top 100
- `REFRESH_SOON` — ranks 101–500
- `MONITOR` — ranks 501–2500
- `SKIP` — everything else

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import numpy as np

# Signal 1: decline (0-100)
decline = (-df['trend_pct'].clip(upper=0)).clip(lower=0, upper=100).fillna(0)

# Signal 2: staleness (0-100, capped at 180 days)
staleness = df['days_since_last_update'].clip(lower=0, upper=180) / 180 * 100

# Signal 3: volume-at-risk (0-100, only if declining)
log_imp = np.log1p(df['impressions_90d'])
volume_at_risk = (log_imp / log_imp.max()) * 100 * (df['trend_pct'] < 0).astype(int)

# Score
df['refresh_score'] = (0.40 * decline + 0.30 * staleness + 0.30 * volume_at_risk).round(2)

# Reason code (one per row)
contrib = pd.DataFrame({
    'DECLINE': decline * 0.40,
    'STALE': staleness * 0.30,
    'VOLUME_AT_RISK': volume_at_risk * 0.30,
})
dominant = contrib.idxmax(axis=1)
share = (contrib.max(axis=1) / contrib.sum(axis=1).replace(0, np.nan)).fillna(0)
total = contrib.sum(axis=1)

df['reason_code'] = np.where(total == 0, 'NO_SIGNAL',
                     np.where(share >= 0.5, dominant, 'MIXED'))

# Action by rank
rank = df['refresh_score'].rank(method='first', ascending=False)
df['action'] = np.select(
    [rank <= 100, rank <= 500, rank <= 2500],
    ['REFRESH_NOW', 'REFRESH_SOON', 'MONITOR'],
    default='SKIP',
)

# Write the queue
queue = df.sort_values('refresh_score', ascending=False)[[
    'content_id', 'refresh_score', 'reason_code', 'action',
    'trend_pct', 'days_since_last_update', 'impressions_90d', 'avg_position',
]]
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Wrote {len(queue)} rows → work/outputs/baseline_action_score.csv")
print("\nAction distribution:")
print(queue['action'].value_counts())
print("\nReason code distribution:")
print(queue['reason_code'].value_counts())
queue.head(10)

Wrote 30000 rows → work/outputs/baseline_action_score.csv

Action distribution:
action
SKIP            27500
MONITOR          2000
REFRESH_SOON      400
REFRESH_NOW       100
Name: count, dtype: int64

Reason code distribution:
reason_code
STALE             10448
DECLINE            7729
MIXED              7610
VOLUME_AT_RISK     4213
Name: count, dtype: int64


,content_id,refresh_score,reason_code,action,trend_pct,days_since_last_update,impressions_90d,avg_position
16751,content_cf56e2e2e282,89.39,MIXED,REFRESH_NOW,-85.6,194,61678,19.7
16514,content_7368877ea310,87.67,MIXED,REFRESH_NOW,-81.5,194,59472,24.8
11489,content_5feee3994adb,86.08,MIXED,REFRESH_NOW,-89.1,194,7812,39.0
7021,content_1bfaa38ff26c,83.03,MIXED,REFRESH_NOW,-74.7,194,25715,22.2
23741,content_df1fa766cac2,81.24,MIXED,REFRESH_NOW,-97.7,304,206,4.8
7509,content_7a888d3d99c8,80.41,MIXED,REFRESH_NOW,-100.0,313,95,67.6
21984,content_02b0d6e30129,80.04,MIXED,REFRESH_NOW,-95.6,313,176,6.9
1659,content_bbca724138f2,79.87,DECLINE,REFRESH_NOW,-100.0,236,75,12.1
7122,content_7a6df559322d,79.57,MIXED,REFRESH_NOW,-94.7,104,43650,0.7
26810,content_ecb6215e79fd,78.90,MIXED,REFRESH_NOW,-74.4,194,4429,25.3


## 3. Top-10 review

One line per row: action, why it's there, what would make it wrong.

1. **content_cf56e2e2e282 — REFRESH_NOW.** High-traffic page (61,678 imp) with steep decline (-85.6%), stale (194d), rank ~20. *Wrong if:* the drop is due to a SERP layout change or query deprecation, not content quality.

2. **content_7368877ea310 — REFRESH_NOW.** Same profile: high traffic (59,472 imp), steep drop (-81.5%), stale (194d), rank ~25. *Wrong if:* the drop comes from an algorithm update outside our control.

3. **content_5feee3994adb — REFRESH_NOW.** Steep drop (-89.1%) and stale, but only 7,812 imp at position 39 (page 4). *Wrong if:* the page has hit its ceiling at page 4 — refreshing won't lift it into page 1.

4. **content_1bfaa38ff26c — REFRESH_NOW.** High traffic (25,715 imp), steep drop (-74.7%), stale (194d). *Wrong if:* the decline is expected seasonality (e.g. a "best of 2024" page that naturally decays).

5. **content_df1fa766cac2 — REFRESH_NOW.** Only 206 impressions, steep drop (-97.7%), stale (304d). Position 4.8 looks good but traffic is tiny. *Wrong if:* absolute traffic is too small for refresh to matter — this is a **weak pick** (see Section 4).

6. **content_7a888d3d99c8 — REFRESH_NOW.** Only 95 impressions, -100% trend, position 67. *Wrong if:* no meaningful traffic to recover — refresh won't move a page ranked #67 into page 1. **Weak pick.**

7. **content_02b0d6e30129 — REFRESH_NOW.** Position 6.9 (page 1!) but only 176 impressions. *Wrong if:* the query behind it has almost no search volume — rank is good, demand is the issue. **Weak pick.**

8. **content_bbca724138f2 — REFRESH_NOW.** 75 impressions, -100% trend, position 12. *Wrong if:* tiny traffic means near-zero business impact. **Weak pick.**

9. **content_7a6df559322d — REFRESH_NOW.** 43,650 impressions, -94.7% trend, position **0.7** (rank #1). *Wrong if:* a rank-#1 page dropping 95% usually signals a SERP feature shift or spam update, not content decay — refreshing content won't fix it.

10. **content_ecb6215e79fd — REFRESH_NOW.** 4,429 impressions, -74.4% trend, position 25.3. *Wrong if:* at page 3, the decline reflects competitiveness the page can't recover without a bigger content strategy.

**Pattern:** top 4 are strong. Rows 5–8 are low-traffic pages that climb the score because of steep declines and staleness — a real weakness of the rule, not a strength.

In [5]:
# Display the top 10 for the review
review_cols = [
    'content_id', 'refresh_score', 'reason_code', 'action',
    'trend_pct', 'days_since_last_update', 'impressions_90d', 'avg_position',
]
queue.head(10)[review_cols].reset_index(drop=True)

,content_id,refresh_score,reason_code,action,trend_pct,days_since_last_update,impressions_90d,avg_position
0,content_cf56e2e2e282,89.39,MIXED,REFRESH_NOW,-85.6,194,61678,19.7
1,content_7368877ea310,87.67,MIXED,REFRESH_NOW,-81.5,194,59472,24.8
2,content_5feee3994adb,86.08,MIXED,REFRESH_NOW,-89.1,194,7812,39.0
3,content_1bfaa38ff26c,83.03,MIXED,REFRESH_NOW,-74.7,194,25715,22.2
4,content_df1fa766cac2,81.24,MIXED,REFRESH_NOW,-97.7,304,206,4.8
5,content_7a888d3d99c8,80.41,MIXED,REFRESH_NOW,-100.0,313,95,67.6
6,content_02b0d6e30129,80.04,MIXED,REFRESH_NOW,-95.6,313,176,6.9
7,content_bbca724138f2,79.87,DECLINE,REFRESH_NOW,-100.0,236,75,12.1
8,content_7a6df559322d,79.57,MIXED,REFRESH_NOW,-94.7,104,43650,0.7
9,content_ecb6215e79fd,78.90,MIXED,REFRESH_NOW,-74.4,194,4429,25.3


## 4. Weak picks + leakage check

### Weak picks

**Rows 5–8** (`content_df1fa766cac2`, `content_7a888d3d99c8`, `content_02b0d6e30129`, `content_bbca724138f2`) are weak picks. All have <250 impressions_90d. They score high because `trend_pct` is very negative (-95% to -100%) and the pages are stale (236–313 days). But the *absolute* traffic is so small that refreshing them will barely move business metrics.

**Why the rule lets them through:** the decline signal is capped at 100, so a -100% drop on a 95-imp page contributes the same 40 points as a -100% drop on a 60,000-imp page. The volume-at-risk signal is log-scaled and weighted at only 0.30 — not enough to demote them.

**What would fix it in the Week-5 model:** either (a) increase volume weight, (b) apply a minimum-impressions filter before scoring, or (c) switch to ranking by *expected recoverable impressions* (a function of traffic and predicted lift) instead of a raw urgency score.

**Row 9** (`content_7a6df559322d`) is a different kind of weak pick: 43,650 impressions, but at position 0.7. A rank-#1 page dropping 95% is usually a SERP-change signal, not a content-quality signal. Refresh action may not help.

### Leakage check

Inputs used by the rule:

- `trend_pct` — past-vs-earlier window (30d vs prev 30d or similar). **Observed, not future.**
- `days_since_last_update` — current snapshot. **Observed.**
- `impressions_90d` — trailing 90-day window. **Observed.**
- `content_id` — identifier only, not scored.

Not used (correctly): `refresh_score`, `reason_code`, `action` are **outputs** of the rule, not inputs. No product flags (e.g. `impression_tier`) were used as features. No label-derived variable was fed back in.

**One note for Week 5:** `trend_pct` overlaps with the outcome we care about (decline). That's fine for a baseline rule. But when you train a predictive model, make sure `trend_pct` is computed only from data **before** the training cutoff, not the same window as the label.

In [6]:
# Confirm no leakage: list the exact columns used as inputs by the rule
print("Inputs used by the rule (observed, past-only):")
print(["trend_pct", "days_since_last_update", "impressions_90d"])

print("\nOutputs written by the rule (not inputs):")
print(["refresh_score", "reason_code", "action"])

print("\nColumns in the raw df that could carry leakage if used:")
print([c for c in df.columns if 'flag' in c.lower() or 'label' in c.lower()])

# Verify weak picks are the ones we identified
weak_ids = ['content_df1fa766cac2', 'content_7a888d3d99c8',
            'content_02b0d6e30129', 'content_bbca724138f2']
weak = queue[queue['content_id'].isin(weak_ids)][
    ['content_id', 'refresh_score', 'impressions_90d', 'trend_pct', 'days_since_last_update']
]
weak.reset_index(drop=True)

Inputs used by the rule (observed, past-only):
['trend_pct', 'days_since_last_update', 'impressions_90d']

Outputs written by the rule (not inputs):
['refresh_score', 'reason_code', 'action']

Columns in the raw df that could carry leakage if used:
[]


,content_id,refresh_score,impressions_90d,trend_pct,days_since_last_update
0,content_df1fa766cac2,81.24,206,-97.7,304
1,content_7a888d3d99c8,80.41,95,-100.0,313
2,content_02b0d6e30129,80.04,176,-95.6,313
3,content_bbca724138f2,79.87,75,-100.0,236


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.